In [1]:
from itertools import combinations
import math

# 16 вариант
coop_game_var_16 = {
    (): 0,
    (0,): 4,
    (1,): 4,
    (2,): 4,
    (3,): 2,
    (0, 1): 9,
    (0, 2): 9,
    (0, 3): 6,
    (1, 2): 10,
    (1, 3): 7,
    (2, 3): 7,
    (0, 1, 2): 15,
    (0, 1, 3): 12,
    (0, 2, 3): 12,
    (1, 2, 3): 14,
    (0, 1, 2, 3): 17,
}


# Проверка супераддитивноси
def is_superadditive(game_values):
    # Проверяем все пары непересекающихся коалиций
    coalitions = list(game_values.keys())
    coalitions_number = len(coalitions)
    
    for coalition_1_index in range(coalitions_number):
        for coalition_2_index in range(coalition_1_index + 1, coalitions_number):
            A = set(coalitions[coalition_1_index])
            B = set(coalitions[coalition_2_index])
            
            # Проверяем только непересекающиеся коалиции
            if A & B:
                continue
            
            # Проверяем, существует ли объединение в словаре
            union = tuple(A | B)
            if union not in game_values:
                continue
            
            # Проверяем условие супераддитивности
            if game_values[union] < game_values[tuple(A)] + game_values[tuple(B)]:
                return False
    
    return True


# Проверка выпуклости
def is_convex(game_values):
    # Проверяем все пары коалиций
    coalitions = list(game_values.keys())
    coalitions_number = len(coalitions)
    
    for coalition_1_index in range(coalitions_number):
        for coalition_2_index in range(coalition_1_index + 1, coalitions_number):
            A = set(coalitions[coalition_1_index])
            B = set(coalitions[coalition_2_index])
            
            union = tuple(A | B)
            intersection = tuple(A & B)
            
            # Проверяем, существуют ли объединение и пересечение в словаре
            if union not in game_values or intersection not in game_values:
                continue
            
            # Проверяем условие выпуклости
            if game_values[tuple(A)] + game_values[tuple(B)] > game_values[union] + game_values[intersection]:
                return False
    
    return True

# Расчет вектора Шепли
def get_shapley_value(game_values):
    # Определяем всех игроков
    all_players = set()
    for coalition in game_values.keys():
        all_players.update(coalition)
    players_amount = len(all_players)
    
    # Словарь для хранения значения Шепли для каждого игрока
    shapley = {player: 0.0 for player in all_players}
    
    # Для каждого игрока вычисляем его вклад
    for player in all_players:
        # Множество остальных игроков
        other_players = [p for p in all_players if p != player]
        other_players_number = len(other_players)
        
        # Перебираем все подмножества остальных игроков
        for k in range(other_players_number + 1):
            # Все комбинации из k игроков
            for subset in combinations(other_players, k):
                S = set(subset)
                
                # Вычисляем предельный вклад игрока
                S_plus_player = tuple(S | {player})
                S_tuple = tuple(S)
                
                marginal_contribution = game_values[S_plus_player] - game_values[S_tuple]
                
                # Весовой коэффициент
                weight = (math.factorial(k) * math.factorial(players_amount - k - 1)) / math.factorial(players_amount)
                
                # Добавляем вклад
                shapley[player] += weight * marginal_contribution
    
    return shapley


# Проверка индивидуальной рационализации
def check_individual_rationality(game_values, shapley_vector):
    # Получаем всех игроков
    all_players = set()
    for coalition in game_values.keys():
        all_players.update(coalition)
    
    for player in all_players:
        individual_value = game_values[(player,)]
        shapley_value = shapley_vector.get(player, 0)
        
        if shapley_value < individual_value:
            False
    
    return True


# Проверка групповой рационализации (эффективности)
def check_coalitional_rationality(game_values, shapley_vector):
    # Получаем всех игроков
    all_players = set()
    for coalition in game_values.keys():
        all_players.update(coalition)
    
    # Суммируем значения Шепли для всех игроков
    total_shapley = sum(shapley_vector.values())
    
    # Получаем значение большой коалиции
    grand_coalition_value = game_values[tuple(all_players)]
    
    # Проверяем условие эффективности
    is_efficient = abs(total_shapley - grand_coalition_value) < 1e-10
    
    return is_efficient

if __name__ == "__main__":

    game_values = coop_game_var_16

    print("Проверка супераддитивности: ", end="")
    if is_superadditive(game_values):
        print("Супераддитивная")
    else:
        print("Нет")

    print("Проверка выпуклости: ", end="")
    if is_convex(game_values):
        print("Выпуклая")
    else:
        print("Нет")
    print()

    shapley_vector = get_shapley_value(game_values)
    print("Вектор Шепли:", shapley_vector)
    print()
    
    print("Проверка индивидуальной рационализации: ", end="")
    if check_individual_rationality(game_values, shapley_vector):
        print("Выполняется")
    else:
        print("Нет")
    
    print("Проверка групповой рационализации: ", end="")
    if check_coalitional_rationality(game_values, shapley_vector):
        print("Выполняется")
    else:
        print("Нет")
    print()


Проверка супераддитивности: Нет
Проверка выпуклости: Нет

Вектор Шепли: {0: 4.166666666666666, 1: 5.166666666666666, 2: 5.166666666666666, 3: 2.5}

Проверка индивидуальной рационализации: Выполняется
Проверка групповой рационализации: Выполняется

